<a href="https://colab.research.google.com/github/Irani-Lutfiani-Putri/Computer-Vision-Klasifikasi-Penyakit-Kulit-Berdasarkan-Fitur-Tekstur-Citra-Menggunakan-Metode-KNN/blob/main/2318023IraniLutfianiPutriKlasifikasiPenyakitKulitMetodeKNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Import Library

In [34]:
import os
import cv2
import numpy as np
import pandas as pd

from skimage.feature import graycomatrix, graycoprops

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

2. Load Dataset

In [35]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Penyakit_Kulit"

class0_path = os.path.join(base_path, "sc_Scabies_sarna")
class1_path = os.path.join(base_path, "ch_Chickenpox_Varicela")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


3. Ekstraksi Fitur GLCM

In [36]:
def extract_glcm_features(folder, label):
    data = []
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        img = cv2.imread(path)
        if img is None: continue

        img = cv2.resize(img, (100,100))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256,
                            symmetric=True, normed=True)

        features = [
            graycoprops(glcm, 'contrast')[0,0],
            graycoprops(glcm, 'dissimilarity')[0,0],
            graycoprops(glcm, 'homogeneity')[0,0],
            graycoprops(glcm, 'energy')[0,0],
            graycoprops(glcm, 'correlation')[0,0],
            graycoprops(glcm, 'ASM')[0,0],
            label
        ]
        data.append(features)
    return data

4. Membuat Dataset & Analisis Fitur

In [37]:
data = []
data += extract_glcm_features(class0_path, 0)
data += extract_glcm_features(class1_path, 1)

df = pd.DataFrame(data, columns=[
    "contrast","dissimilarity","homogeneity",
    "energy","correlation","ASM","Class"
])

print(df.head())
print(df.describe())

df.to_csv("/content/drive/MyDrive/Penyakit_Kulit/hasil_fitur_penyakitkulit.csv", index=False)

     contrast  dissimilarity  homogeneity    energy  correlation       ASM  \
0   74.326061       3.911313     0.407255  0.058062     0.981328  0.003371   
1   73.494343       5.898586     0.202761  0.020014     0.973157  0.000401   
2  231.768990       9.558889     0.125661  0.019084     0.836696  0.000364   
3   44.766970       4.714040     0.263775  0.049884     0.992713  0.002488   
4  265.176364       7.173333     0.382971  0.210793     0.969713  0.044434   

   Class  
0      0  
1      0  
2      0  
3      0  
4      0  
          contrast  dissimilarity  homogeneity      energy  correlation  \
count   613.000000     613.000000   613.000000  613.000000   613.000000   
mean    174.784651       6.679545     0.261188    0.050995     0.931945   
std     166.164102       3.125814     0.109638    0.068605     0.075745   
min       5.044242       1.123636     0.046822    0.012448     0.302491   
25%      57.720404       4.404949     0.177002    0.022795     0.915144   
50%     116.123

5. Preprocessing & Split Data

In [38]:
X = df.drop("Class", axis=1)
y = df["Class"]

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

6. Tuning & Model KNN

In [39]:
for k in range(1,11):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    print("K =", k, "| Accuracy =", accuracy_score(y_test, knn.predict(X_test)))

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

K = 1 | Accuracy = 0.6097560975609756
K = 2 | Accuracy = 0.6016260162601627
K = 3 | Accuracy = 0.5284552845528455
K = 4 | Accuracy = 0.5691056910569106
K = 5 | Accuracy = 0.5609756097560976
K = 6 | Accuracy = 0.5691056910569106
K = 7 | Accuracy = 0.5528455284552846
K = 8 | Accuracy = 0.5691056910569106
K = 9 | Accuracy = 0.5609756097560976
K = 10 | Accuracy = 0.5772357723577236


KNeighborsClassifier(n_neighbors=3)

7. Prediksi & Evaluasi

In [40]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5284552845528455
[[25 33]
 [25 40]]
              precision    recall  f1-score   support

           0       0.50      0.43      0.46        58
           1       0.55      0.62      0.58        65

    accuracy                           0.53       123
   macro avg       0.52      0.52      0.52       123
weighted avg       0.53      0.53      0.52       123

